# 🧠 Build Your Own Brain — GDG on Campus HWU
### A Retrieval-Augmented Generation (RAG) Starter Notebook

Welcome! By the end of **Session 1**, you'll have built a working search engine over real text — first on a shared example, then on a topic of *your own choosing*.

**Today we are only building retrieval (search).** Generation (turning retrieved text into a spoken answer) is Session 2.

Run the cells top to bottom. Don't skip the setup cells — they install and configure everything we need.


## 🔧 Step 0 — Setup
Run this first. It installs everything we need for today.

In [ ]:
!pip install -q google-genai numpy wikipedia pypdf requests
print("✅ Packages installed.")


### Get your free Gemini API key

You should have already created this **before today** (see the pre-session email). If not:
1. Go to https://aistudio.google.com/apikey
2. Sign in with your Google account
3. Click "Create API key" and copy it

Paste it below when prompted. It stays private to your own Colab session — we never see it, and it's never written to this notebook file.

In [ ]:
from getpass import getpass

API_KEY = getpass("Paste your Gemini API key here and press Enter: ")

from google import genai
client = genai.Client(api_key=API_KEY)

print("✅ Client ready.")


---
## 📘 PART 1 — Learn the pipeline on a shared example

We'll build **chunk → embed → retrieve** together, step by step, using one shared demo topic: the **Burj Khalifa**.

Once this works, Part 2 asks you to point the exact same code at a topic *you* picked.


### Step 1 — The Universal Loader

RAG doesn't care *what* your text is about — it only cares that you can get it into one plain string. This function can pull text from four kinds of sources:

| `source_type` | `source` should be... | Notes |
|---|---|---|
| `"wikipedia"` | an exact article title, e.g. `"Burj Khalifa"` | Cleanest option — use this when you can |
| `"gdoc"` | a Google Doc link or ID | Doc must be shared as "Anyone with the link can view" |
| `"pdf"` | a file path to an uploaded `.pdf` | Text-based PDFs only — **not** scanned/image PDFs |
| `"txt"` | a file path to an uploaded `.txt` | Always the safest fallback |
| `"paste"` | raw text you paste directly as a string | For anything copy-pasted by hand (e.g. a niche wiki page) |

It also automatically checks the size of what you've loaded and trims it if it's too big — this keeps every team comfortably inside the free API tier's rate limits, regardless of topic.

In [ ]:
import re
import requests

# --- Size guardrails ---
# These numbers are chosen deliberately: MAX_CHARS keeps us at roughly 25-40 chunks,
# which is enough for retrieval to be meaningful without burning through free-tier
# API rate limits during a live session. MIN_CHARS is a floor below which there's
# not really enough material for "search" to mean anything.
MIN_CHARS = 1500
MAX_CHARS = 15000


def _extract_gdoc_id(url_or_id):
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", url_or_id)
    return match.group(1) if match else url_or_id


def _load_wikipedia(title):
    import wikipedia
    wikipedia.set_lang("en")
    page = wikipedia.page(title, auto_suggest=False)
    return page.content


def _load_gdoc(url_or_id):
    doc_id = _extract_gdoc_id(url_or_id)
    export_url = f"https://docs.google.com/document/d/{doc_id}/export?format=txt"
    resp = requests.get(export_url)
    resp.raise_for_status()
    return resp.text


def _load_pdf(path):
    from pypdf import PdfReader
    reader = PdfReader(path)
    pages_text = [page.extract_text() or "" for page in reader.pages]
    text = "\n".join(pages_text)
    if not text.strip():
        raise ValueError(
            "No text could be extracted. This is likely a scanned/image PDF, "
            "which this workshop doesn't support — pick a text-based PDF instead."
        )
    return text


def _load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_text(source_type, source):
    """Universal loader: returns clean plain text regardless of source type,
    and automatically enforces a safe size range for the free API tier."""
    loaders = {
        "wikipedia": _load_wikipedia,
        "gdoc": _load_gdoc,
        "pdf": _load_pdf,
        "txt": _load_txt,
        "paste": lambda s: s,
    }
    if source_type not in loaders:
        raise ValueError(f"Unknown source_type '{source_type}'. Choose one of: {list(loaders.keys())}")

    text = loaders[source_type](source).strip()
    char_count = len(text)
    print(f"Loaded {char_count:,} characters.")

    if char_count < MIN_CHARS:
        print(
            f"⚠️  That's quite short (under {MIN_CHARS:,} chars). Retrieval needs a bit more "
            f"material to actually search over — consider a longer article or section."
        )

    if char_count > MAX_CHARS:
        text = text[:MAX_CHARS]
        print(
            f"⚠️  Trimmed from {char_count:,} to {MAX_CHARS:,} characters to stay safely inside "
            f"the free API tier's rate limits. Tip: pick ONE focused article/section, not an "
            f"entire wiki or book."
        )

    return text


### Step 2 — Load the shared demo dataset

In [ ]:
demo_text = load_text("wikipedia", "Burj Khalifa")
print(demo_text[:500], "...")


---
### Step 3 — Chunking

Why chunk? Two reasons: the model can only usefully focus on a limited amount of text at once, and **smaller, well-sized chunks make retrieval more precise** — if a chunk is too big, you'll retrieve a lot of irrelevant text along with the relevant sentence.

We're using a simple **fixed-size chunker with overlap**: cut every `chunk_size` characters, but let consecutive chunks overlap slightly so we don't accidentally slice a sentence in half at a chunk boundary.

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


demo_chunks = chunk_text(demo_text)
print(f"Split into {len(demo_chunks)} chunks.")
print("\n--- Example chunk ---\n")
print(demo_chunks[0])


---
### Step 4 — Embedding

An embedding turns text into a list of numbers that captures *meaning* — texts with similar meaning end up with similar numbers. We're using Google's `gemini-embedding-001` model.

We embed in small batches with a short pause between them — this is deliberate, to stay well under the free tier's requests-per-minute limit rather than firing everything at once.

In [ ]:
import time
import numpy as np


def embed_texts(client, texts, batch_size=10, delay_seconds=2):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        for attempt in range(3):
            try:
                result = client.models.embed_content(model="gemini-embedding-001", contents=batch)
                all_embeddings.extend([e.values for e in result.embeddings])
                break
            except Exception as e:
                print(f"  Hit an error ({e}), retrying in 5s...")
                time.sleep(5)
        print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks...")
        time.sleep(delay_seconds)
    return np.array(all_embeddings)


demo_embeddings = embed_texts(client, demo_chunks)
print(f"\n✅ Done. Embedding matrix shape: {demo_embeddings.shape}")


---
### Step 5 — Retrieval

Given a question, we embed it the same way, then find which chunks are "closest" in meaning using **cosine similarity** — a measure of how aligned two vectors are. Highest similarity = most relevant chunk.

This is the actual mechanism underneath every vector database — we're just doing it by hand, in plain NumPy, so you can see exactly what's happening.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)


def retrieve(client, query, chunks, chunk_embeddings, top_k=3):
    query_result = client.models.embed_content(model="gemini-embedding-001", contents=query)
    query_embedding = np.array(query_result.embeddings[0].values)

    scores = [cosine_similarity(query_embedding, emb) for emb in chunk_embeddings]
    ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]


In [ ]:
# Try it! Ask a question about the Burj Khalifa.
query = "How tall is the Burj Khalifa?"

results = retrieve(client, query, demo_chunks, demo_embeddings, top_k=3)

for rank, (chunk, score) in enumerate(results, start=1):
    print(f"#{rank}  (similarity: {score:.3f})")
    print(chunk)
    print("---")


**Try breaking it.** Ask a question that *isn't* covered in the article (e.g. about a completely different building) and look at the similarity scores — low scores across the board are the system honestly telling you "I don't have a good answer for this." That's a real and important signal, not a bug.

In [ ]:
query = "Try your own question here"

results = retrieve(client, query, demo_chunks, demo_embeddings, top_k=3)
for rank, (chunk, score) in enumerate(results, start=1):
    print(f"#{rank}  (similarity: {score:.3f})")
    print(chunk)
    print("---")


---
## 🎉 Checkpoint

You just built the full **chunk → embed → retrieve** pipeline — the "search" half of RAG — from scratch. Next session we'll connect this to an AI model so it can actually *answer* using what we retrieve, instead of just returning raw chunks.

**Part 2 is where it gets personal: same pipeline, your topic.**


---
## 📗 PART 2 — Build it on YOUR topic

Pick one of the four source types from Step 1, fill in the two variables below, and re-run the exact same pipeline — nothing else changes.

**Reminder:** one focused article/section, not an entire wiki or book. The loader will warn you (and auto-trim) if you go over, but a tighter topic = better results and faster embedding.

In [ ]:
# ---- EDIT THESE TWO LINES ----
MY_SOURCE_TYPE = "wikipedia"        # one of: "wikipedia", "gdoc", "pdf", "txt", "paste"
MY_SOURCE = "Your topic here"       # article title / gdoc link / file path / pasted text
# -------------------------------

my_text = load_text(MY_SOURCE_TYPE, MY_SOURCE)
my_chunks = chunk_text(my_text)
print(f"\n{len(my_chunks)} chunks ready to embed.")


In [ ]:
my_embeddings = embed_texts(client, my_chunks)
print(f"\n✅ Done. Embedding matrix shape: {my_embeddings.shape}")


In [ ]:
# Ask it something!
my_query = "Ask your bot a question about your own topic"

results = retrieve(client, my_query, my_chunks, my_embeddings, top_k=3)
for rank, (chunk, score) in enumerate(results, start=1):
    print(f"#{rank}  (similarity: {score:.3f})")
    print(chunk)
    print("---")


### Before you go

Try **3 questions** on your own topic:
1. One you're confident it should nail
2. One that's a bit tricky/ambiguous
3. One that's completely outside the topic (to see the "I don't know" signal in action)

Note which one surprised you most — we'll talk about it at the start of Session 2, where we turn this raw retrieval into an actual spoken answer, and look at how to make retrieval more accurate.

**See you next session — bring your laptop with this notebook saved (File → Save a copy in Drive).**
